# 5. Flat-Baseline Spike Inference & Tempered SMC Evidence (`sample_smc`)

This tutorial replaces `docs/notebooks/05_Fitting_Flat.ipynb`, where `ticktack 1.x` fitted a pure spike on a flat cosmic-ray baseline (`spike_only`) to `Miyake12_Cedar` (`Guttler15`). Because `v1`'s adaptive Runge–Kutta solver stepped clean across narrow sub-annual spikes whenever the baseline had zero solar derivative, `v1` had to disable adaptive integration (`adaptive=False`) and force `oversample=1008` fixed sub-steps per year, taking **`26.9 s`** in `emcee` while providing no Bayesian evidence $\ln Z$ to test whether solar modulation is warranted.

`ticktack 2.0`'s `ExactLinearSolver` splits integration sub-intervals analytically at the exact `MiyakeSpike` support knots via clamped [Van Loan (1978)](https://doi.org/10.1109/TAC.1978.1101743) matrix exponentials—conserving fluence to machine precision down to day-scale pulses with zero oversampling—while `BlackJAX` adaptive tempered `sample_smc` delivers calibrated marginal log-evidences for both flat-baseline and `HarmonicSolar` models.

In [ ]:
import math, time
import jax, jax.numpy as jnp, numpy as np
import ticktack as tt

cedar = tt.TreeRingSeries.from_csv(
    "notebooks/miyake12.csv", hemisphere="north", growth_season="april-september", name="Miyake12_Cedar"
)
ds = tt.Dataset(cedar, name="Miyake12_Flat")
cbm = tt.load_preset("Guttler15")
flat_model = tt.RadiocarbonModel(
    cbm, tt.SteadyProduction() + tt.MiyakeSpike(t_start=774.5, duration=0.25, fluence=6.5, profile="boxcar")
)
narrow_jump = float(jnp.max(flat_model.set("production.spike.duration", 0.005).predict_d14c(ds)))
broad_jump = float(jnp.max(flat_model.predict_d14c(ds)))
print(f"ExactLinearSolver peak jump without oversampling: dur=0.25 yr -> {broad_jump:.2f} permil, dur=0.005 yr -> {narrow_jump:.2f} permil")

## Adaptive Tempered SMC on the Flat-Baseline Model vs `v1` (`26.9 s`)

Defining normalized unconstrained priors in `ParameterSpace` allows `sample_smc` to anneal particles from inverse temperature `beta = 0` (exact prior) to `beta = 1` (posterior) with multinomial resampling and [Lee & Whiteley (2018)](https://doi.org/10.1093/biomet/asy028) ancestral-lineage variance estimation (which yields a consistent single-run estimator of the normalizing-constant variance under multinomial resampling), returning both posterior samples and an unbiased log-evidence estimate.

In [ ]:
specs = [
    tt.ParamSpec("production.spike.t_start", "sigmoid", 0.0, 1.2, low=773.5, high=776.0),
    tt.ParamSpec("production.spike.duration", "sigmoid", -0.5, 1.2, low=0.02, high=2.0),
    tt.ParamSpec("production.spike.fluence", "exp", math.log(6.5), 0.5),
]
ps_flat = tt.ParameterSpace(flat_model, specs)
cfg_flat = tt.LinearMarginalConfig(offset_mode="shared", marginalize_solar=False)
t0 = time.perf_counter()
smc_flat = tt.sample_smc(flat_model, ds, ps_flat, cfg_flat, num_particles=400, num_mcmc_steps=5, rng_key=jax.random.PRNGKey(501))
jax.block_until_ready(smc_flat.log_evidence)
dt_smc_cold = time.perf_counter() - t0
t1 = time.perf_counter()
smc_flat_w = tt.sample_smc(flat_model, ds, ps_flat, cfg_flat, num_particles=400, num_mcmc_steps=5, rng_key=jax.random.PRNGKey(502))
jax.block_until_ready(smc_flat_w.log_evidence)
dt_smc_warm = time.perf_counter() - t1
print(f"Flat-baseline ln Z = {float(smc_flat.log_evidence):.2f} +/- {float(smc_flat.log_evidence_err):.2f} | cold={dt_smc_cold:.2f} s ({26.9/dt_smc_cold:.1f}x vs 26.9 s v1), warm={dt_smc_warm:.2f} s ({26.9/dt_smc_warm:.1f}x)")

## Bayesian Model Comparison: Flat Baseline vs `HarmonicSolar` ($\Delta\ln\widehat{Z}$)

Adding `HarmonicSolar(periods=11.0)` with analytical quadrature marginalization (`solar_prior_sd=0.35`) tests directly via the Bayes factor $\Delta\ln\widehat{Z} = \ln\widehat{Z}_{\mathrm{solar}} - \ln\widehat{Z}_{\mathrm{flat}}$ whether the 28 cedar rings require an 11-year solar cycle on top of the 774 CE spike.

In [ ]:
solar_model = tt.RadiocarbonModel(
    cbm, tt.ProductionSum(
        steady=tt.SteadyProduction(), solar=tt.HarmonicSolar(periods=11.0, t_ref=775.0),
        spike=tt.MiyakeSpike(t_start=774.5, duration=0.25, fluence=6.5, profile="boxcar"),
    ),
)
ps_solar = tt.ParameterSpace(solar_model, specs)
cfg_solar = tt.LinearMarginalConfig(offset_mode="shared", solar_prior_sd=0.35, marginalize_solar=True)
smc_solar = tt.sample_smc(solar_model, ds, ps_solar, cfg_solar, num_particles=400, num_mcmc_steps=5, rng_key=jax.random.PRNGKey(503))
d_lnz = float(smc_solar.log_evidence - smc_flat.log_evidence)
d_err = math.hypot(float(smc_solar.log_evidence_err), float(smc_flat.log_evidence_err))
fl_flat = smc_flat.idata.posterior["production__spike__fluence"].values.reshape(-1)
print(f"HarmonicSolar ln Z = {float(smc_solar.log_evidence):.2f} +/- {float(smc_solar.log_evidence_err):.2f} | Delta ln Z (solar - flat) = {d_lnz:+.2f} +/- {d_err:.2f}\nFlat SMC fluence = {fl_flat.mean():.2f} +/- {fl_flat.std():.2f} atoms/cm^2/s*yr ({smc_flat.tempering_betas.size - 1} tempering bridges)")

## Flat-Baseline SMC Posterior $\Delta^{14}\mathrm{C}(t)$ Trajectory and Pulls

Evaluating continuous atmospheric $\Delta^{14}\mathrm{C}(t)$ curves across a subset of resampled SMC posterior particles illustrates how cleanly the three-parameter flat-baseline pulse captures the 774–775 CE cedar excursion.

In [ ]:
t_fine = jnp.linspace(759.5, 788.5, 291)
u_sub = smc_flat.particles[:120]
d14c_smc = jax.vmap(lambda u: tt.linear_posterior(ps_flat.constrain(u, flat_model), ds, cfg_flat, t_fine=t_fine).d14c_fine_mean)(u_sub)
u_mean = jnp.mean(smc_flat.particles, axis=0)
lp_flat = tt.linear_posterior(ps_flat.constrain(u_mean, flat_model), ds, cfg_flat, t_fine=t_fine)
fig_fit, axes_fit = tt.plot_d14c_fit(
    ds,
    t_fine=t_fine,
    d14c_pred=d14c_smc,
    ring_pred=lp_flat.ring_pred_mean,
    show_residuals=True,
    title="Flat-Baseline SMC Posterior Fit & Standardized Pulls (Guttler15)",
)

## One-to-One Observed vs Predicted Calibration

Plotting the flat-baseline posterior mean ring predictions against `Miyake12_Cedar` observations on `plot_data_model_correlation` confirms a well-calibrated $\chi^2 / N_{\mathrm{obs}}$ with no systematic curvature across pre-spike or post-spike rings.

In [ ]:
fig_corr, ax_corr = tt.plot_data_model_correlation(
    ds,
    lp_flat.ring_pred_mean,
    title="Flat-Baseline SMC Fit: Observed vs Predicted Calibration",
)

Clamped Van Loan sub-interval quadrature (`ExactLinearSolver`) eliminates `v1`'s `adaptive=False, oversample=1008` constant-step workaround completely, while `BlackJAX` tempered SMC (`sample_smc`) computes calibrated thermodynamic Bayes factors $\Delta\ln\widehat{Z}$ between flat-baseline and solar-modulated cosmic-ray models in a fraction of `v1`'s `26.9 s` `emcee` runtime.